<a href="https://colab.research.google.com/github/nafeessiddique-dev/XAIApproxRL/blob/main/Updated_mobilenetv2_gradcam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch torchvision torchaudio --no-cache-dir

In [ ]:
!pip install torch-xla
!pip install opencv-python
!pip install numpy torchao
!pip install ninja

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 4.0 MB/s eta 0:00:00


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import cv2
import matplotlib.pyplot as plt
import os
from torchvision import models, datasets, transforms
from PIL import Image
import shutil
import math
from torch.utils.cpp_extension import load
import time
from scipy.stats import spearmanr

# =============================================================================
# MAC CONFIGURATIONS - ALL EVOAPPROX MULTIPLIERS
# =============================================================================
MAC_CONFIGURATIONS = {
    'mul8s_1KV6_add16se_2TN': {'multiplier': 'mul8s_1KV6', 'adder': 'add16se_2TN', 'mul_power_mW': 0.425, 'mul_delay_ns': 1.48, 'add_power_mW': 0.072, 'add_delay_ns': 0.50, 'mae': 0.00, 'mre': 0.00, 'level': 0},
    'mul8s_1KV8_add16se_2TN': {'multiplier': 'mul8s_1KV8', 'adder': 'add16se_2TN', 'mul_power_mW': 0.422, 'mul_delay_ns': 1.48, 'add_power_mW': 0.072, 'add_delay_ns': 0.50, 'mae': 0.0018, 'mre': 0.28, 'level': 1},
    'mul8s_1KV9_add16se_2TN': {'multiplier': 'mul8s_1KV9', 'adder': 'add16se_2TN', 'mul_power_mW': 0.410, 'mul_delay_ns': 1.47, 'add_power_mW': 0.072, 'add_delay_ns': 0.50, 'mae': 0.0064, 'mre': 0.90, 'level': 2},
    'mul8s_1KVA_add16se_2TN': {'multiplier': 'mul8s_1KVA', 'adder': 'add16se_2TN', 'mul_power_mW': 0.391, 'mul_delay_ns': 0.89, 'add_power_mW': 0.072, 'add_delay_ns': 0.50, 'mae': 0.019, 'mre': 2.53 , 'level': 3},
    'mul8s_1KVP_add16se_2TN': {'multiplier': 'mul8s_1KVP', 'adder': 'add16se_2TN', 'mul_power_mW': 0.363, 'mul_delay_ns': 1.37, 'add_power_mW': 0.072, 'add_delay_ns': 0.50, 'mae': 0.051, 'mre': 2.73, 'level': 4},
    'mul8s_1KVQ_add16se_2TN': {'multiplier': 'mul8s_1KVQ', 'adder': 'add16se_2TN', 'mul_power_mW': 0.351, 'mul_delay_ns': 0.89, 'add_power_mW': 0.072, 'add_delay_ns': 0.50, 'mae': 0.056, 'mre': 3.64, 'level': 5},
    'mul8s_1KX5_add16se_2TN': {'multiplier': 'mul8s_1KX5', 'adder': 'add16se_2TN', 'mul_power_mW': 0.289, 'mul_delay_ns': 0.89, 'add_power_mW': 0.072, 'add_delay_ns': 0.50, 'mae': 0.15, 'mre': 8.93, 'level': 6},
    'mul8s_1L2J_add16se_2TN': {'multiplier': 'mul8s_1L2J', 'adder': 'add16se_2TN', 'mul_power_mW': 0.301, 'mul_delay_ns': 1.36, 'add_power_mW': 0.072, 'add_delay_ns': 0.50, 'mae': 0.081, 'mre': 4.41, 'level': 7},
    'mul8s_1L2L_add16se_2TN': {'multiplier': 'mul8s_1L2L', 'adder': 'add16se_2TN', 'mul_power_mW': 0.200, 'mul_delay_ns': 1.14, 'add_power_mW': 0.072, 'add_delay_ns': 0.50, 'mae': 0.23, 'mre': 12.26, 'level': 8},
    'mul8s_1L2N_add16se_2TN': {'multiplier': 'mul8s_1L2N', 'adder': 'add16se_2TN', 'mul_power_mW': 0.126, 'mul_delay_ns': 0.94, 'add_power_mW': 0.072, 'add_delay_ns': 0.50, 'mae': 0.52, 'mre': 27.44, 'level': 9},
    'mul8s_1L12_add16se_2TN': {'multiplier': 'mul8s_1L12', 'adder': 'add16se_2TN', 'mul_power_mW': 0.052, 'mul_delay_ns': 0.89, 'add_power_mW': 0.072, 'add_delay_ns': 0.50, 'mae': 3.08, 'mre': 135.77, 'level': 10}
}
# =============================================================================
# ROHNAS HARDWARE PARAMETERS
# =============================================================================
HARDWARE_CONFIG = {
    'clock_period_ns':          3.0,
    'pe_array_size':            65536,
    'load_weights_cycles':      100,
    'sram_energy_pJ_per_access': 1.92,
}

# =============================================================================
# RL REWARD WEIGHTS
# =============================================================================
REWARD_WEIGHTS = {
    'energy_efficiency':      0.40,
    'spearman_correlation':   0.10,
    'top_pixel_concentration': 0.50,
}

FEATURE_SIMILARITY_THRESHOLD = 0.15

# =============================================================================
# OXFORD-IIIT PET CLASS MAP  (37 cat & dog breeds, label index 0-36 → name)
# torchvision.datasets.OxfordIIITPet — stable API since torchvision 0.12:
#   dataset._images  → list[Path]   (absolute paths to .jpg files)
#   dataset._labels  → list[int]    (0-based class indices)
# =============================================================================
OXFORD_PETS_CLASSES = {
    0:  'Abyssinian',          1:  'American Bulldog',      2:  'American Pit Bull Terrier',
    3:  'Basset Hound',        4:  'Beagle',                5:  'Bengal',
    6:  'Birman',              7:  'Bombay',                8:  'Boxer',
    9:  'British Shorthair',  10:  'Chihuahua',            11:  'Egyptian Mau',
   12:  'English Cocker Spaniel', 13: 'English Setter',   14:  'German Shorthaired',
   15:  'Great Pyrenees',     16:  'Havanese',             17:  'Japanese Chin',
   18:  'Keeshond',           19:  'Leonberger',           20:  'Maine Coon',
   21:  'Miniature Pinscher', 22:  'Newfoundland',         23:  'Persian',
   24:  'Pomeranian',         25:  'Pug',                  26:  'Ragdoll',
   27:  'Russian Blue',       28:  'Saint Bernard',        29:  'Samoyed',
   30:  'Scottish Terrier',   31:  'Shiba Inu',            32:  'Siamese',
   33:  'Sphynx',             34:  'Staffordshire Bull Terrier', 35: 'Wheaten Terrier',
   36:  'Yorkshire Terrier',
}

# =============================================================================
# ADAPT LINEAR LAYERS  (from the MobileNetV2 reference script)
# =============================================================================

class AdaPT_Linear_Python_Function(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input, weight, bias, bias_):
        ctx.save_for_backward(input, weight, bias)
        ctx.bias_ = bias_
        quant_input  = torch.clamp(input  * 127.0, -128, 127).to(torch.int8)
        quant_weight = torch.clamp(weight * 127.0, -128, 127).to(torch.int8)
        quant_input_approx  = (quant_input  >> 2) << 2
        quant_weight_approx = (quant_weight >> 2) << 2
        output = torch.matmul(
            quant_input_approx.to(torch.float32),
            quant_weight_approx.to(torch.float32).t()
        ) / (127.0 * 127.0)
        if bias_:
            output += bias
        return output

    @staticmethod
    def backward(ctx, grad_output):
        input, weight, bias = ctx.saved_tensors
        grad_input = grad_weight = grad_bias = None
        if ctx.needs_input_grad[0]:
            grad_input  = grad_output.mm(weight)
        if ctx.needs_input_grad[1]:
            grad_weight = grad_output.t().mm(input)
        if ctx.bias_ and ctx.needs_input_grad[2]:
            grad_bias   = grad_output.sum(0)
        return grad_input, grad_weight, grad_bias, None


class AdaPT_Linear_Python(nn.Module):
    def __init__(self, size_in, size_out, bias=True, axx_mult='mul8s_1L2L'):
        super().__init__()
        self.size_in, self.size_out, self.bias_ = size_in, size_out, bias
        self.weight = nn.Parameter(torch.Tensor(size_out, size_in))
        self.bias   = nn.Parameter(torch.Tensor(size_out)) if bias else None
        self.axx_mult = axx_mult
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        if self.bias is not None:
            fan_in, _ = nn.init._calculate_fan_in_and_fan_out(self.weight)
            bound = 1 / math.sqrt(fan_in)
            nn.init.uniform_(self.bias, -bound, bound)

    def forward(self, x):
        return AdaPT_Linear_Python_Function.apply(x, self.weight, self.bias, self.bias_)

# =============================================================================
# IMAGE QUALITY FILTER — CONTRAST & OBJECT FOCUS
# =============================================================================
IMAGE_FILTER_CONFIG = {
    'contrast_threshold':   0.10,   # Lowered from 0.12 - accept lower contrast
    'center_weight_ratio':  0.35,   # Lowered from 0.40 - accept less centered subjects
    'center_crop_fraction': 0.50,   # size of the "centre" window (fraction of W/H)
    'landscape_only':       True,   # Only keep landscape (width >= height) images
}


def compute_rms_contrast(img_array: np.ndarray) -> float:
    """RMS contrast — std-dev of grayscale luminance (values in [0,1])."""
    gray = (0.299 * img_array[:, :, 0] +
            0.587 * img_array[:, :, 1] +
            0.114 * img_array[:, :, 2])
    return float(np.std(gray))


def compute_center_weight_ratio(img_array: np.ndarray,
                                center_fraction: float = 0.50) -> float:
    """Fraction of total image energy inside the central crop.
    High value → bright subject is centred rather than clipped to an edge."""
    H, W, _ = img_array.shape
    energy_full = np.sum(img_array ** 2)
    if energy_full == 0:
        return 0.0
    h_margin = int(H * (1 - center_fraction) / 2)
    w_margin = int(W * (1 - center_fraction) / 2)
    crop = img_array[max(h_margin, 0):min(H - h_margin, H),
                     max(w_margin, 0):min(W - w_margin, W), :]
    return float(np.sum(crop ** 2) / energy_full)


def passes_quality_filter(img_array: np.ndarray,
                           config: dict = IMAGE_FILTER_CONFIG) -> tuple:
    """Return (passes, reason_str, metrics_dict)."""
    H, W, _ = img_array.shape

    contrast    = compute_rms_contrast(img_array)
    center_ratio = compute_center_weight_ratio(img_array, config['center_crop_fraction'])

    contrast_ok      = contrast     >= config['contrast_threshold']
    center_ratio_ok  = center_ratio >= config['center_weight_ratio']

    # Check orientation if landscape_only is enabled
    is_landscape = W >= H
    landscape_ok = not config.get('landscape_only', False) or is_landscape

    metrics = {
        'rms_contrast':        round(contrast,     4),
        'center_weight_ratio': round(center_ratio, 4),
        'width':               W,
        'height':              H,
        'is_landscape':        is_landscape,
    }

    if contrast_ok and center_ratio_ok and landscape_ok:
        return True, 'PASS', metrics

    reasons = []
    if not contrast_ok:
        reasons.append(f"low contrast ({contrast:.4f} < {config['contrast_threshold']})")
    if not center_ratio_ok:
        reasons.append(f"object not centred ({center_ratio:.4f} < {config['center_weight_ratio']})")
    if not landscape_ok:
        reasons.append(f"portrait orientation ({W}×{H}, need landscape)")
    return False, ' | '.join(reasons), metrics


def filter_images(images: list, config: dict = IMAGE_FILTER_CONFIG, label: str = '') -> list:
    """Keep only images that pass both quality criteria, with a printed report."""
    kept = []
    print(f"\n🔍 Filtering {len(images)} {label}images for contrast & object focus…")
    print(f"   Thresholds → contrast ≥ {config['contrast_threshold']} | "
          f"centre ratio ≥ {config['center_weight_ratio']}")
    if config.get('landscape_only', False):
        print(f"   Orientation → LANDSCAPE ONLY (width ≥ height)")

    for img_data in images:
        ok, reason, metrics = passes_quality_filter(img_data['array'], config)
        tag  = '✅ KEEP' if ok else '❌ DROP'
        name = img_data.get('class_name', os.path.basename(str(img_data.get('path', '?'))))
        orientation = '🖼️ LAND' if metrics.get('is_landscape', True) else '📱 PORT'
        print(f"  {tag}  {orientation}  {name:<30}  "
              f"{metrics.get('width', 0)}×{metrics.get('height', 0)}  "
              f"contrast={metrics['rms_contrast']:.4f}  "
              f"centre={metrics['center_weight_ratio']:.4f}"
              + (f"  → {reason}" if not ok else ''))
        if ok:
            img_data['quality_metrics'] = metrics
            kept.append(img_data)

    print(f"\n  Kept {len(kept)}/{len(images)} images ({len(images)-len(kept)} dropped).")
    return kept

# =============================================================================
# CONTEXTUAL BANDIT FOR ONLINE RL
# =============================================================================

class ContextualBandit:
    """Contextual Bandit for Online Learning."""

    def __init__(self, mac_config_names, tau=0.5):
        self.configs   = mac_config_names
        self.tau       = tau
        self.Q         = {c: 0.5 for c in mac_config_names}
        self.N         = {c: 0   for c in mac_config_names}
        self.history   = []
        self.context_Q = {}
        self.context_N = {}

    def get_context_key(self, context_features):
        mi = context_features['mean_intensity']
        sp = context_features['sparsity']
        intensity_bin = 'low' if mi < 0.3 else ('medium' if mi < 0.6 else 'high')
        sparsity_bin  = 'sparse' if sp < 0.15 else 'dense'
        return f"{intensity_bin}_{sparsity_bin}"

    def select_action(self, context_features, mode='greedy'):
        context_key = self.get_context_key(context_features)
        q_values = np.array([
            self.context_Q.get((context_key, c), self.Q[c])
            for c in self.configs
        ])
        if mode == 'random':
            return np.random.choice(self.configs)
        if mode == 'greedy':
            return self.configs[int(np.argmax(q_values))]
        # softmax
        exp_v = np.exp(q_values / self.tau)
        probs = exp_v / np.sum(exp_v)
        return self.configs[np.random.choice(len(self.configs), p=probs)]

    def update(self, context_features, action, reward):
        context_key = self.get_context_key(context_features)
        n = self.N[action]
        self.Q[action] = (n * self.Q[action] + reward) / (n + 1)
        self.N[action] = n + 1
        key = (context_key, action)
        if key in self.context_Q:
            nc = self.context_N[key]
            self.context_Q[key] = (nc * self.context_Q[key] + reward) / (nc + 1)
            self.context_N[key] = nc + 1
        else:
            self.context_Q[key] = reward
            self.context_N[key] = 1
        self.history.append({'context': context_features, 'context_key': context_key,
                             'action': action, 'reward': reward})

    def print_summary(self):
        print("\n" + "="*70)
        print("CONTEXTUAL BANDIT LEARNING SUMMARY")
        print("="*70)
        print("\nGlobal Q-values:")
        for c in sorted(self.configs, key=lambda x: self.Q[x], reverse=True):
            print(f"  {c:<30} Q={self.Q[c]:.4f}  N={self.N[c]}")
        print(f"\nContext-specific policies learned: "
              f"{len(set(k[0] for k in self.context_Q))}")
        print(f"Total experiences: {len(self.history)}")

# =============================================================================
# GRADIENT MAP FEATURE EXTRACTION
# =============================================================================

def extract_gradient_features(gradient_map_normalized):
    flat = gradient_map_normalized.flatten()
    mean_intensity = float(np.mean(flat))
    sorted_g = np.sort(flat)[::-1]
    n_top    = max(1, int(len(sorted_g) * 0.10))
    top_sum  = np.sum(sorted_g[:n_top])
    total    = np.sum(sorted_g)
    sparsity = float(top_sum / total) if total > 0 else 0.0
    hist, _  = np.histogram(flat, bins=5, range=(0, 1))
    return {
        'mean_intensity': mean_intensity,
        'sparsity':       sparsity,
        'histogram':      (hist / len(flat)).tolist(),
    }


def are_features_similar(f1, f2, threshold=FEATURE_SIMILARITY_THRESHOLD):
    dist = (abs(f1['mean_intensity'] - f2['mean_intensity']) +
            abs(f1['sparsity']       - f2['sparsity'])) / 2.0
    return dist < threshold

# =============================================================================
# METRICS
# =============================================================================

def calculate_spearman_correlation(map1, map2):
    a, b = map1.flatten(), map2.flatten()
    if len(a) != len(b) or np.std(a) == 0 or np.std(b) == 0:
        return 0.0
    corr, _ = spearmanr(a, b)
    return 0.0 if np.isnan(corr) else float(corr)


def calculate_top_pixel_concentration(gradient_map, top_percentage=0.10):
    abs_g  = np.abs(gradient_map.flatten())
    sorted_g = np.sort(abs_g)[::-1]
    n_top  = max(1, int(len(sorted_g) * top_percentage))
    total  = np.sum(sorted_g)
    return 0.0 if total == 0 else float(np.sum(sorted_g[:n_top]) / total)

# =============================================================================
# GRAD-CAM HEATMAP VISUALISATION HELPERS
# =============================================================================

def create_heatmap_jet(cam_map_normalized, original_size):
    """Resize CAM → JET colourmap, returns BGR uint8 for cv2.imwrite."""
    resized = cv2.resize(cam_map_normalized, original_size, interpolation=cv2.INTER_CUBIC)
    g8      = np.uint8(255 * np.clip(resized, 0, 1))
    return cv2.applyColorMap(g8, cv2.COLORMAP_JET)           # BGR


def create_heatmap_overlay(cam_map_normalized, img_rgb_uint8, original_size,
                            alpha: float = 0.45):
    """
    Blend JET heatmap over the original image (alpha on heatmap).
    img_rgb_uint8: (H, W, 3) uint8 RGB array.
    Returns BGR uint8 for cv2.imwrite.
    """
    img_bgr  = cv2.cvtColor(
        cv2.resize(img_rgb_uint8, original_size, interpolation=cv2.INTER_CUBIC),
        cv2.COLOR_RGB2BGR
    )
    heatmap  = create_heatmap_jet(cam_map_normalized, original_size)
    return cv2.addWeighted(heatmap, alpha, img_bgr, 1.0 - alpha, 0)


# =============================================================================
# UNIVERSAL ROI DETECTION  (from reference Grad-CAM script)
# =============================================================================

def detect_roi_region(img_rgb):
    """Detect the primary object bounding box, then refine to focus on face/head region.
    Optimized for landscape pet images - finds full body then crops to head area."""
    h, w = img_rgb.shape[:2]
    best_bbox  = None
    method_used = "none"

    # Method 1: OpenCV saliency
    try:
        saliency = cv2.saliency.StaticSaliencyFineGrained_create()
        success, sal_map = saliency.computeSaliency(img_rgb)
        if success:
            sal_map = (sal_map * 255).astype(np.uint8)

            # Use adaptive threshold based on saliency distribution
            thresh_val = max(0.5 * sal_map.max(), np.percentile(sal_map, 80))
            _, thresh  = cv2.threshold(sal_map, int(thresh_val), 255, cv2.THRESH_BINARY)

            # Morphological operations to connect regions
            kernel     = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
            thresh     = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
            thresh     = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel)

            contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

            # Score contours based on size and centrality
            candidates = []
            for cnt in sorted(contours, key=cv2.contourArea, reverse=True)[:5]:
                x, y, bw, bh = cv2.boundingRect(cnt)
                area_ratio = (bw * bh) / (w * h)

                # Good size range for pet subjects
                if 0.05 < area_ratio < 0.90:
                    # Calculate how centered the bbox is
                    center_x = (x + bw/2) / w
                    center_y = (y + bh/2) / h
                    center_score = 1.0 - (abs(center_x - 0.5) + abs(center_y - 0.5))

                    # Prefer larger, more centered regions
                    area_score = min(area_ratio / 0.4, 1.0)
                    total_score = center_score * 0.4 + area_score * 0.6

                    candidates.append((total_score, x, y, bw, bh))

            if candidates:
                candidates.sort(reverse=True)
                _, x, y, bw, bh = candidates[0]
                best_bbox = (x, y, bw, bh)
                method_used = "saliency"
    except Exception as e:
        print(f"  ⚠ Saliency detection failed: {e}")

    # Method 2: Edge detection
    if best_bbox is None:
        try:
            gray    = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

            # Enhance contrast
            gray = cv2.equalizeHist(gray)

            blurred = cv2.GaussianBlur(gray, (5, 5), 0)
            med     = np.median(blurred)
            edges   = cv2.Canny(blurred, int(max(0, 0.66 * med)), int(min(255, 1.33 * med)))

            # More aggressive dilation to connect pet features
            kernel  = cv2.getStructuringElement(cv2.MORPH_RECT, (9, 9))
            edges   = cv2.dilate(edges, kernel, iterations=3)
            edges   = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel)

            contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

            candidates = []
            for cnt in sorted(contours, key=cv2.contourArea, reverse=True)[:8]:
                x, y, bw, bh = cv2.boundingRect(cnt)
                area_ratio = (bw * bh) / (w * h)

                if 0.08 < area_ratio < 0.90:
                    center_x = (x + bw/2) / w
                    center_y = (y + bh/2) / h
                    center_score = 1.0 - (abs(center_x - 0.5) + abs(center_y - 0.5))

                    area_score = min(area_ratio / 0.4, 1.0)
                    total_score = center_score * 0.4 + area_score * 0.6

                    candidates.append((total_score, x, y, bw, bh))

            if candidates:
                candidates.sort(reverse=True)
                _, x, y, bw, bh = candidates[0]
                best_bbox = (x, y, bw, bh)
                method_used = "edges"
        except Exception as e:
            print(f"  ⚠ Edge detection failed: {e}")

    # Refine bbox to focus on FACE/HEAD region (upper portion)
    if best_bbox is not None:
        x, y, bw, bh = best_bbox

        # First, do modest expansion to ensure we have full subject
        exp = 0.15
        x_exp  = max(0, int(x - bw * exp))
        y_exp  = max(0, int(y - bh * exp))
        bw_exp = min(w - x_exp, int(bw * (1 + 2 * exp)))
        bh_exp = min(h - y_exp, int(bh * (1 + 2 * exp)))

        # Now crop to upper portion (face/head region)
        # For pets, face is typically in upper 40-50% of body bbox
        face_height_ratio = 0.50  # Use upper 50% of detected region
        face_y_start = y_exp
        face_height = int(bh_exp * face_height_ratio)

        # Keep horizontal extent but may expand slightly for head width
        face_width_ratio = 0.85  # Use central 85% horizontally
        face_x_margin = int(bw_exp * (1 - face_width_ratio) / 2)
        face_x_start = x_exp + face_x_margin
        face_width = int(bw_exp * face_width_ratio)

        # Ensure we stay within image bounds
        face_x_start = max(0, face_x_start)
        face_y_start = max(0, face_y_start)
        face_width = min(w - face_x_start, face_width)
        face_height = min(h - face_y_start, face_height)

        print(f"  ✓ ROI via {method_used.upper()}: Full=({x_exp},{y_exp},{bw_exp},{bh_exp}) → Face=({face_x_start},{face_y_start},{face_width},{face_height})")
        return (face_x_start, face_y_start, face_width, face_height)

    # Fallback: upper-center crop (where pet faces typically are)
    cr_w = 0.60  # 60% width
    cr_h = 0.45  # 45% height
    x = int(w * (1 - cr_w) / 2)
    y = int(h * 0.15)  # Start 15% from top (not centered vertically)
    crop_w = int(w * cr_w)
    crop_h = int(h * cr_h)
    print(f"  → ROI: upper-center crop ({x},{y},{crop_w},{crop_h}) [FACE-FOCUSED FALLBACK]")
    return (x, y, crop_w, crop_h)


def apply_focused_mask(cam, region_bbox, img_shape,
                        focus_strength=0.95, focus_mode='hard'):
    """Multiply CAM by a face-focused mask that emphasizes the upper portion.
    Creates a gradient that's strongest at the top (eyes/ears) and fades downward."""
    h, w      = img_shape[:2]
    x, y, bw, bh = region_bbox
    mask      = np.zeros((h, w), dtype=np.float32)

    # Create base mask for ROI
    mask[y:y + bh, x:x + bw] = 1.0

    # Create VERTICAL GRADIENT within the ROI (stronger at top for face focus)
    # This emphasizes eyes, ears, forehead - the most expressive facial features
    gradient = np.zeros((h, w), dtype=np.float32)
    for i in range(y, min(y + bh, h)):
        # Position within ROI (0 at top, 1 at bottom)
        pos_in_roi = (i - y) / bh if bh > 0 else 0
        # Exponential decay: very strong at top (face), weaker at bottom
        gradient_value = np.exp(-3.0 * pos_in_roi)  # e^(-3*x) creates strong top emphasis
        gradient[i, x:x+bw] = gradient_value

    # Combine base mask with gradient
    mask = mask * gradient

    # Apply blur based on focus mode
    if focus_mode == 'hard':
        ks   = max(3, int(min(bw, bh) * 0.08)) | 1
        mask = cv2.GaussianBlur(mask, (ks, ks), 0)
        mask = np.power(mask, 0.5)  # Mild power to preserve gradient shape
    elif focus_mode == 'soft':
        ks   = int(min(bw, bh) * 0.15) | 1
        mask = cv2.GaussianBlur(mask, (ks, ks), 0)
    elif focus_mode == 'strict':
        # Very tight focus with minimal blur
        ks   = max(3, int(min(bw, bh) * 0.05)) | 1
        mask = cv2.GaussianBlur(mask, (ks, ks), 0)
        mask = np.power(mask, 0.7)

    # Normalize mask
    if mask.max() > 0:
        mask = (mask - mask.min()) / (mask.max() - mask.min() + 1e-8)

    # Apply focus strength with extra power for tight focus
    if focus_strength > 0.9:
        mask = np.power(mask, 1.3)  # Sharper falloff

    mask = mask * focus_strength

    # Resize and apply to CAM
    mask_r = cv2.resize(mask, (cam.shape[1], cam.shape[0]), interpolation=cv2.INTER_CUBIC)
    cam_focused = cam * mask_r

    # Normalize focused CAM
    if cam_focused.max() > 0:
        cam_focused = (cam_focused - cam_focused.min()) / (cam_focused.max() - cam_focused.min() + 1e-8)

    return cam_focused

# =============================================================================
# LAYER DESCRIPTOR & PROFILER
# =============================================================================

class LayerDescriptor:
    def __init__(self, layer_type, name, input_shape, output_shape,
                 kernel_size=1, stride=1, params=None):
        self.layer_type  = layer_type
        self.name        = name
        self.input_shape = input_shape
        self.output_shape = output_shape
        self.kernel_size = kernel_size
        self.stride      = stride
        self.params      = params or {}
        if layer_type == 'conv':
            self.chin, self.hin, self.win   = input_shape
            self.chout, self.hout, self.wout = output_shape
            self.nin  = self.hin
            self.nout = self.hout
        elif layer_type == 'fc':
            self.chin  = input_shape[0]  if isinstance(input_shape,  tuple) else input_shape
            self.chout = output_shape[0] if isinstance(output_shape, tuple) else output_shape
            self.nin   = 1
            self.nout  = 1

    def get_weights_count(self):
        if self.layer_type == 'conv':
            return self.kernel_size * self.kernel_size * self.chin * self.chout
        if self.layer_type == 'fc':
            return self.chin * self.chout
        return 0

    def get_sum_count(self):
        if self.layer_type == 'conv':
            return self.kernel_size * self.kernel_size * self.chin
        if self.layer_type == 'fc':
            return self.chin
        return 0

    def get_feature_maps_count(self):
        if self.layer_type == 'conv':
            return self.nout * self.nout
        if self.layer_type == 'fc':
            return 1
        return 0


class LayerProfiler:
    def __init__(self, model, input_shape=(1, 3, 224, 224)):
        self.model             = model
        self.input_shape       = input_shape
        self.layer_descriptors = []
        self.hooks             = []

    def _register_hooks(self):
        def hook_fn(name):
            def hook(module, inp, output):
                inp = inp[0] if isinstance(inp, tuple) else inp
                in_shape  = tuple(inp.shape[1:])
                out_shape = tuple(output.shape[1:])
                if isinstance(module, nn.Conv2d):
                    ks = module.kernel_size[0] if isinstance(module.kernel_size, tuple) else module.kernel_size
                    st = module.stride[0]       if isinstance(module.stride,      tuple) else module.stride
                    self.layer_descriptors.append(LayerDescriptor(
                        'conv', name, in_shape, out_shape, kernel_size=ks, stride=st))
                elif isinstance(module, (nn.Linear, AdaPT_Linear_Python)):
                    chin  = in_shape[0]  if len(in_shape)  == 1 else int(np.prod(in_shape))
                    chout = out_shape[0] if len(out_shape) == 1 else int(np.prod(out_shape))
                    self.layer_descriptors.append(LayerDescriptor(
                        'fc', name, (chin,), (chout,)))
            return hook

        for name, module in self.model.named_modules():
            if isinstance(module, (nn.Conv2d, nn.Linear, AdaPT_Linear_Python)):
                self.hooks.append(module.register_forward_hook(hook_fn(name)))

    def profile(self):
        self._register_hooks()
        dummy = torch.randn(*self.input_shape)
        self.model.eval()
        with torch.no_grad():
            self.model(dummy)
        for h in self.hooks:
            h.remove()
        return self.layer_descriptors

# =============================================================================
# ROHNAS ENERGY MODEL
# =============================================================================

class RoHNASEnergyModel:
    def __init__(self, mac_config_name):
        cfg = MAC_CONFIGURATIONS[mac_config_name]
        self.config_name          = mac_config_name
        self.approximation_level  = cfg['level']
        self.mac_energy_pJ        = cfg['mul_power_mW'] + cfg['add_power_mW']
        self.T                    = HARDWARE_CONFIG['clock_period_ns']
        self.pe_array_size        = HARDWARE_CONFIG['pe_array_size']
        self.E_memory             = HARDWARE_CONFIG['sram_energy_pJ_per_access']

    def calculate_layer_energy_rohnas(self, layer_desc):
        wl = layer_desc.get_weights_count()
        sl = layer_desc.get_sum_count()
        fl = layer_desc.get_feature_maps_count()
        wPEarray = math.ceil(wl / (self.pe_array_size * min(self.pe_array_size, sl)))
        m_acc    = 65536 if fl == 1 else self.pe_array_size * max(sl - 15, 1)
        cl       = wl * wPEarray + fl
        memory_energy_pJ   = (m_acc / 128.0) * self.E_memory
        pe_array_energy_pJ = cl * self.mac_energy_pJ
        return cl * self.T, memory_energy_pJ + pe_array_energy_pJ

    def calculate_ig_energy(self, n_steps, layer_descriptors):
        total_pJ = sum(self.calculate_layer_energy_rohnas(l)[1] for l in layer_descriptors)
        ig_pJ    = total_pJ * (n_steps * 2)
        return {'ig_energy_mJ': ig_pJ / 1e9, 'config_name': self.config_name,
                'approximation_level': self.approximation_level}

# =============================================================================
# MAC ENERGY MODEL AND LAYER ENERGY TRACKER
# =============================================================================

class MACEnergyModel:
    """Energy model for MAC operations based on hardware configuration."""
    def __init__(self, mac_config_name):
        cfg = MAC_CONFIGURATIONS[mac_config_name]
        self.config_name = mac_config_name
        self.mul_power_mW = cfg['mul_power_mW']
        self.add_power_mW = cfg['add_power_mW']
        self.mul_delay_ns = cfg['mul_delay_ns']
        self.add_delay_ns = cfg['add_delay_ns']
        self.mac_energy_pJ = self.mul_power_mW + self.add_power_mW
        self.approximation_level = cfg['level']
        self.mre = cfg['mre']

    def get_mac_energy_pJ(self):
        return self.mac_energy_pJ


class LayerEnergyTracker:
    """Tracks energy consumption across model layers during forward pass."""
    def __init__(self, energy_model: MACEnergyModel):
        self.energy_model = energy_model
        self.layer_energies = {}
        self.total_macs = 0
        self.total_energy_pJ = 0.0

    def reset(self):
        """Reset all tracked values."""
        self.layer_energies = {}
        self.total_macs = 0
        self.total_energy_pJ = 0.0

    def count_conv2d(self, layer_name, input_shape, weight_shape, stride, padding):
        """Count MACs for a Conv2D layer."""
        try:
            batch_size, in_channels, in_h, in_w = input_shape
            out_channels, _, kernel_h, kernel_w = weight_shape

            # Calculate output dimensions
            out_h = (in_h + 2 * padding - kernel_h) // stride + 1
            out_w = (in_w + 2 * padding - kernel_w) // stride + 1

            # MACs = output_pixels × kernel_ops_per_pixel
            macs = out_h * out_w * out_channels * in_channels * kernel_h * kernel_w * batch_size
            energy_pJ = macs * self.energy_model.get_mac_energy_pJ()

            self.layer_energies[layer_name] = {
                'macs': macs,
                'energy_pJ': energy_pJ,
                'type': 'conv2d'
            }
            self.total_macs += macs
            self.total_energy_pJ += energy_pJ
        except Exception as e:
            pass  # Silently skip problematic layers

    def count_linear(self, layer_name, batch_size, in_features, out_features):
        """Count MACs for a Linear/FC layer."""
        try:
            macs = batch_size * in_features * out_features
            energy_pJ = macs * self.energy_model.get_mac_energy_pJ()

            self.layer_energies[layer_name] = {
                'macs': macs,
                'energy_pJ': energy_pJ,
                'type': 'linear'
            }
            self.total_macs += macs
            self.total_energy_pJ += energy_pJ
        except Exception as e:
            pass  # Silently skip problematic layers

    def get_summary(self):
        """Return summary of energy consumption."""
        return {
            'total_macs': self.total_macs,
            'total_energy_pJ': self.total_energy_pJ,
            'total_energy_mJ': self.total_energy_pJ / 1e9,
            'config_name': self.energy_model.config_name,
            'approximation_level': self.energy_model.approximation_level,
            'num_layers': len(self.layer_energies)
        }

# =============================================================================
# APPROXIMATION FUNCTIONS
# =============================================================================

def load_approximate_multiplier_kernel(axx_mult):
    try:
        return load(
            name=f'PyInit_approx_mult_{axx_mult}',
            sources=["/content/adapt/cpu-kernels/axx_linear.cpp"],
            extra_cflags=[f'-DAXX_MULT={axx_mult} -march=native -fopenmp -O3'],
            extra_include_paths=["/content/adapt/cpu-kernels"],
            extra_ldflags=['-lgomp'],
            verbose=False,
        )
    except Exception as e:
        print(f"  ✗ Failed to load {axx_mult}: {e}")
        raise


# =============================================================================
# GRAD-CAM COMPUTATION
# Replaces Integrated Gradients. The C++ approximate-multiplier kernels are
# used in the model's AdaPT_Linear_Python layers (loaded once per config in
# the RL loop via load_approximate_multiplier_kernel). The gradient
# approximation is additionally applied inside the Grad-CAM backward hook
# to simulate approximate multiplier behaviour on the backprop path,
# exactly mirroring the reference Grad-CAM script.
# =============================================================================

# Grad-CAM target: last MobileNetV2 feature block before global avg-pool
_GRADCAM_LAYER_NAMES = [
    'base_model.model.features.18',  # For wrapped models
    'base_model.model.features.17',
    'base_model.model.features.16',
    'model.features.18',              # For unwrapped models
    'model.features.17',
    'model.features.16',
]


def _find_gradcam_layer(model: nn.Module) -> nn.Module:
    named = dict(model.named_modules())
    for name in _GRADCAM_LAYER_NAMES:
        if name in named:
            return named[name]
    # Debug: print available layer names to help troubleshoot
    available = [n for n in named.keys() if 'features' in n and any(str(i) in n for i in [16, 17, 18])]
    raise ValueError(f"No Grad-CAM layer found. Tried: {_GRADCAM_LAYER_NAMES}\nAvailable feature layers: {available[:10]}")


def apply_gradient_approximation(grad: torch.Tensor, mre: float) -> torch.Tensor:
    """
    Quantise + noise gradients proportional to multiplier MRE.
    Matches the reference Grad-CAM script exactly.
    """
    if mre < 0.01:
        return grad
    if   mre < 0.5:  scale = 64.0
    elif mre < 1.5:  scale = 32.0
    elif mre < 3.5:  scale = 16.0
    elif mre < 8.0:  scale =  8.0
    elif mre < 20.0: scale =  4.0
    elif mre < 50.0: scale =  2.0
    else:            scale =  1.0
    grad_q = torch.round(grad * scale) / scale
    noise  = torch.randn_like(grad_q) * (mre / 100.0) * 0.05
    return grad_q + noise


def compute_grad_cam(model: nn.Module,
                     img_tensor: torch.Tensor,
                     energy_tracker,
                     mre: float = 0.0,
                     target_class: int = None,
                     focus_on_region: bool = True,
                     focus_strength: float = 0.95,
                     focus_mode: str = 'hard',
                     img_rgb_uint8: np.ndarray = None) -> tuple:
    """
    Compute Grad-CAM for one image.

    The energy_tracker is reset and populated during the forward pass so
    energy accounting matches the reference script exactly.

    Args:
        model           : MobileNetV2Classifier (eval mode)
        img_tensor      : normalised (C,H,W) tensor
        energy_tracker  : LayerEnergyTracker — reset + filled here
        mre             : multiplier MRE; controls gradient approximation
        target_class    : class to explain; None → argmax
        focus_on_region : apply ROI-focused mask after computing CAM
        focus_strength  : mask blend strength (0–1)
        focus_mode      : 'hard' | 'soft' | 'strict'
        img_rgb_uint8   : (H,W,3) uint8 RGB original for ROI detection

    Returns:
        cam_np       : np.ndarray (H,W) in [0,1], size = 224×224
        cam_original : np.ndarray (H,W) in [0,1], resized to original_size
                       (or same as cam_np if img_rgb_uint8 is None)
        target_class : int
        pred_class   : int
        pred_conf    : float
        energy_summary : dict from energy_tracker.get_summary()
    """
    conv_layer  = _find_gradcam_layer(model)
    image_batch = img_tensor.unsqueeze(0)

    activations, gradients = {}, {}

    def fwd_hook(m, inp, out):
        activations['value'] = out.detach()
        out.retain_grad()

    def bwd_hook(m, grad_in, grad_out):
        g = grad_out[0]
        if mre > 0.0:
            g = apply_gradient_approximation(g, mre)
        gradients['value'] = g.detach()

    hf = conv_layer.register_forward_hook(fwd_hook)
    hb = conv_layer.register_full_backward_hook(bwd_hook)

    try:
        # Reset energy tracker BEFORE forward pass (matches reference)
        energy_tracker.reset()
        model.eval()

        with torch.set_grad_enabled(True):
            output     = model(image_batch)
            output     = torch.clamp(output, -88, 88)
            pred_class = int(output.argmax(1).item())
            pred_conf  = float(torch.exp(output[0, pred_class]).item())

            if target_class is None:
                target_class = pred_class

            loss = F.nll_loss(output, torch.tensor([target_class]))
            model.zero_grad()
            loss.backward()

        act     = activations['value'][0]      # (C, h, w)
        grad    = gradients['value'][0]        # (C, h, w)
        weights = torch.mean(grad, dim=(1, 2)) # (C,)
        if mre > 0.0:
            weights = apply_gradient_approximation(
                weights.unsqueeze(0).unsqueeze(0), mre
            ).squeeze()

        cam = torch.sum(weights.view(-1, 1, 1) * act, dim=0)
        cam = F.relu(cam)
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        cam = F.interpolate(
            cam.unsqueeze(0).unsqueeze(0), (224, 224),
            mode='bilinear', align_corners=False
        ).squeeze()
        cam_np = cam.cpu().numpy()   # (224, 224) in [0,1]

        # ── ROI focus ──────────────────────────────────────────────────────
        if focus_on_region and img_rgb_uint8 is not None:
            original_size = (img_rgb_uint8.shape[1], img_rgb_uint8.shape[0])
            detected_roi  = detect_roi_region(img_rgb_uint8)
            cam_orig      = cv2.resize(cam_np, original_size, interpolation=cv2.INTER_CUBIC)
            cam_orig      = apply_focused_mask(cam_orig, detected_roi,
                                               img_rgb_uint8.shape,
                                               focus_strength, focus_mode)
        else:
            cam_orig = cam_np
            detected_roi = None

        energy_summary = energy_tracker.get_summary()
        return cam_np, cam_orig, target_class, pred_class, pred_conf, energy_summary

    finally:
        hf.remove()
        hb.remove()

# =============================================================================
# LOAD OXFORD-IIIT PET DATASET
# =============================================================================

def load_oxford_pets_dataset(num_images: int = 30):
    """
    Download and load the Oxford-IIIT Pet Dataset via torchvision.

    Why Oxford-IIIT Pets for XAI gradient visualisation?
      • Every image is a professional close-up of a single animal with the
        subject tightly cropped and centred → consistently high centre-energy
        ratio AND high RMS contrast.
      • Coat colours/textures (tabby stripes, golden fur, brindle, etc.)
        produce strong, localised gradient signals that make XAI maps
        visually sharp and meaningful.
      • 37 fine-grained breeds (25 dog + 12 cat) with ~200 images each give
        enough per-class variation to stress the RL multiplier selector.
      • torchvision.datasets.OxfordIIITPet has a fully stable public API
        since torchvision 0.12:
            dataset._images  → list[Path]   (absolute paths to .jpg files)
            dataset._labels  → list[int]    (0-based class indices 0-36)

    Each returned dict contains:
        array         – float32 (224, 224, 3) in [0,1], ready for the model
        path          – pathlib.Path to original file (for high-res reload)
        source        – 'oxford_pets'
        original_size – (W, H) of the raw file
        class_name    – human-readable breed name
        label         – integer class index 0-36
    """
    print("📥 Loading Oxford-IIIT Pet Dataset…")
    data_root = '/content/datasets'

    transform_model = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ])

    try:
        dataset = datasets.OxfordIIITPet(
            root=data_root,
            split='test',          # 3,669 images — smaller, loads fast
            target_types='category',
            download=True,
            transform=transform_model,
        )
        print(f"  ✓ torchvision.datasets.OxfordIIITPet loaded  "
              f"({len(dataset)} images, 37 breeds)")
    except Exception as e:
        raise RuntimeError(
            f"Failed to load OxfordIIITPet: {e}\n"
            "Ensure torchvision ≥ 0.12 is installed."
        )

    # Verify the stable path attribute
    if not hasattr(dataset, '_images'):
        raise AttributeError(
            "torchvision.datasets.OxfordIIITPet does not expose '_images'. "
            f"Available attributes: {[a for a in dir(dataset) if not a.startswith('__')]}"
        )

    np.random.seed(42)
    indices = np.random.choice(len(dataset), min(num_images, len(dataset)), replace=False)

    images_data = []
    for idx in indices:
        img_tensor, label = dataset[int(idx)]

        img_path      = dataset._images[int(idx)]                # pathlib.Path
        img_pil       = Image.open(img_path).convert('RGB')
        original_size = img_pil.size                             # (W, H)
        img_np        = img_tensor.permute(1, 2, 0).numpy()     # (224, 224, 3)
        class_name    = OXFORD_PETS_CLASSES.get(label, f'breed_{label}')

        images_data.append({
            'array':         img_np,
            'img_rgb_uint8': (img_np * 255).astype(np.uint8),   # for ROI + overlay
            'path':          img_path,
            'source':        'oxford_pets',
            'original_size': original_size,
            'class_name':    class_name,
            'label':         label,
        })

    print(f"✓ Loaded {len(images_data)} images from Oxford-IIIT Pets")
    print(f"  Breeds: {', '.join(sorted(set(d['class_name'] for d in images_data)))}")
    return images_data

# =============================================================================
# SETUP — COPY C++ KERNELS
# =============================================================================

src_dir = '/content/drive/MyDrive/adapt-main/adapt/cpu-kernels'
dst_dir = '/content/adapt/cpu-kernels'
os.makedirs(dst_dir, exist_ok=True)
os.makedirs(f'{dst_dir}/axx_mults', exist_ok=True)

print("\n" + "="*80)
print("ONLINE RL — OXFORD-IIIT PETS DATASET — MOBILENETV2 — GRAD-CAM HEATMAPS")
print("="*80)

print("\n📋 Copying C++ files…")
try:
    shutil.copy(f'{src_dir}/axx_linear.cpp', f'{dst_dir}/axx_linear.cpp')
    for cfg in MAC_CONFIGURATIONS.values():
        shutil.copy(
            f'{src_dir}/axx_mults/{cfg["multiplier"]}.h',
            f'{dst_dir}/axx_mults/{cfg["multiplier"]}.h',
        )
    print("✓ C++ files copied")
except FileNotFoundError as e:
    print(f"✗ Error copying files: {e}")

# =============================================================================
# MODEL — MobileNetV2 + AdaPT_Linear_Python classifier
# =============================================================================
device = torch.device('cpu')


class MobileNetV2Classifier(nn.Module):
    """
    MobileNetV2 with the final classifier layer replaced by AdaPT_Linear_Python.
    num_classes is inferred from the checkpoint at load time — do not hardcode it here.
    """
    def __init__(self, num_classes=10):
        super().__init__()
        self.model = models.mobilenet_v2(weights=None)
        in_features = self.model.classifier[1].in_features
        self.model.classifier[1] = AdaPT_Linear_Python(
            in_features, num_classes, bias=True, axx_mult='mul8s_1L2L'
        )

    def forward(self, x):
        return F.log_softmax(self.model(x), dim=1)


class MobileNetV2WithEnergyTracking(nn.Module):
    """
    Wraps MobileNetV2Classifier with forward-pass hooks that feed
    LayerEnergyTracker, matching the reference Grad-CAM script approach.
    The AdaPT_Linear_Python classifier layer is already loaded from the
    checkpoint and its C++ kernel is embedded in the layer's forward call.
    """
    def __init__(self, base_model: nn.Module, tracker):
        super().__init__()
        self.base_model     = base_model
        self.energy_tracker = tracker
        self.hooks          = []
        self._register_hooks()

    def _register_hooks(self):
        def conv_hook(name):
            def hook(m, inp, out):
                if not self.training and self.energy_tracker and len(inp) > 0:
                    try:
                        self.energy_tracker.count_conv2d(
                            name, inp[0].shape, m.weight.shape,
                            m.stride[0]   if isinstance(m.stride,   tuple) else m.stride,
                            m.padding[0]  if isinstance(m.padding,  tuple) else m.padding,
                        )
                    except Exception:
                        pass
            return hook

        def linear_hook(name):
            def hook(m, inp, out):
                if not self.training and self.energy_tracker and len(inp) > 0:
                    try:
                        w = m.weight.shape
                        self.energy_tracker.count_linear(name, inp[0].shape[0], w[1], w[0])
                    except Exception:
                        pass
            return hook

        def adapt_hook(name):
            def hook(m, inp, out):
                if not self.training and self.energy_tracker and len(inp) > 0:
                    try:
                        w = m.weight.shape
                        self.energy_tracker.count_linear(name, inp[0].shape[0], w[1], w[0])
                    except Exception:
                        pass
            return hook

        for name, module in self.base_model.named_modules():
            if isinstance(module, nn.Conv2d):
                self.hooks.append(module.register_forward_hook(conv_hook(name)))
            elif isinstance(module, nn.Linear):
                self.hooks.append(module.register_forward_hook(linear_hook(name)))
            elif isinstance(module, AdaPT_Linear_Python):
                self.hooks.append(module.register_forward_hook(adapt_hook(name)))

    def forward(self, x):
        return self.base_model(x)

    def cleanup_hooks(self):
        for h in self.hooks:
            h.remove()
        self.hooks = []
# num_classes is auto-detected from the checkpoint — no need to match manually.
checkpoint_path = '/content/mobilenetv2_checkpoint.pth'

if not os.path.exists(checkpoint_path):
    print(f"✗ Error: Checkpoint not found at {checkpoint_path}")
    exit()

print(f"\n📦 Loading MobileNetV2 model from {checkpoint_path}…")
checkpoint = torch.load(checkpoint_path, map_location='cpu')

# ── Auto-detect num_classes from the checkpoint so the architecture always
#    matches regardless of which dataset the model was trained on.
#    The classifier weight has shape [num_classes, in_features].
_state   = checkpoint['model_state_dict']
_clf_key = 'model.classifier.1.weight'
if _clf_key not in _state:
    _clf_key = 'classifier.1.weight'   # some checkpoints omit the outer 'model.' wrapper
if _clf_key not in _state:
    raise KeyError(
        f"Cannot find classifier weight in checkpoint. "
        f"Keys (first 10): {list(_state.keys())[:10]}"
    )
num_classes_ckpt = _state[_clf_key].shape[0]
print(f"  ℹ Detected num_classes = {num_classes_ckpt} from checkpoint ('{_clf_key}')")

if num_classes_ckpt != 102:
    print(f"  ⚠ Checkpoint has {num_classes_ckpt} output classes (not 102).")
    print( "    Model will run as-is. To use a Oxford Pets model, retrain with num_classes=37.")

model = MobileNetV2Classifier(num_classes=num_classes_ckpt)
model.load_state_dict(_state)
model.to(device)
model.eval()
print(f"✓ MobileNetV2 loaded  (num_classes={num_classes_ckpt})")

profiler          = LayerProfiler(model, input_shape=(1, 3, 224, 224))
layer_descriptors = profiler.profile()
print(f"✓ Extracted {len(layer_descriptors)} layers")

# =============================================================================
# ENERGY NORMALISATION — LayerEnergyTracker per MAC config
# (mirrors the reference Grad-CAM script: LayerEnergyTracker counts MACs
#  via forward hooks during each Grad-CAM call; here we do one dry-run
#  to record E_max for the accurate baseline config)
# =============================================================================
print("\n📊 Computing energy normalisation values…")

all_energy_mJ = {}
for config_name in MAC_CONFIGURATIONS:
    _em      = MACEnergyModel(config_name)
    _tracker = LayerEnergyTracker(_em)
    _wrapped = MobileNetV2WithEnergyTracking(model, _tracker)
    _wrapped.eval()
    with torch.no_grad():
        _wrapped(torch.randn(1, 3, 224, 224))
    _wrapped.cleanup_hooks()
    all_energy_mJ[config_name] = _tracker.get_summary()['total_energy_mJ']

E_max = max(all_energy_mJ.values())
print(f"  E_max (accurate baseline) = {E_max:.6f} mJ")

# =============================================================================
# LOAD AND FILTER OXFORD-IIIT PET IMAGES
# =============================================================================
print("\n📸 Loading Oxford-IIIT Pets images…")
# Load a larger pool (100 images) so enough landscape images survive the quality filter
oxford_pets_images_raw = load_oxford_pets_dataset(num_images=100)

# Load optional external images with better error handling
external_image_paths = [
    '/content/1.jpg',
    '/content/2.jpg',
    '/content/3.jpg',
]

external_images_raw = []
for ext_path in external_image_paths:
    if os.path.exists(ext_path):
        try:
            img           = Image.open(ext_path).convert('RGB')
            original_size = img.size
            img_resized   = img.resize((224, 224), Image.BILINEAR)
            img_array     = np.array(img_resized).astype(np.float32) / 255.0

            # Determine a more descriptive name from the file
            filename = os.path.basename(ext_path)
            class_name = filename.replace('.jpg', '').replace('.png', '').replace('_', ' ').title()

            external_images_raw.append({
                'array':         img_array,
                'img_rgb_uint8': (img_array * 255).astype(np.uint8),
                'path':          ext_path,
                'source':        'external',
                'original_size': original_size,
                'class_name':    class_name,
                'label':         -1,
            })
            print(f"  ✓ Loaded external: {filename} ({original_size[0]}×{original_size[1]})")
        except Exception as e:
            print(f"  ✗ Failed to load {ext_path}: {e}")
    else:
        print(f"  ✗ Not found: {ext_path}")

# ── quality filtering ───────────────────────────────────────────────────────
oxford_pets_images = filter_images(oxford_pets_images_raw, IMAGE_FILTER_CONFIG, label='Oxford Pets ')
external_images    = filter_images(external_images_raw,    IMAGE_FILTER_CONFIG, label='external ')

# Combine and cap at 15 images for the RL loop
all_images = (oxford_pets_images + external_images)[:15]

print(f"\n✓ Total images after filtering: {len(all_images)}")
print(f"  - Oxford Pets (kept): {len(oxford_pets_images)}")
print(f"  - External    (kept): {len(external_images)}")

if len(all_images) == 0:
    print("\n⚠️  No images passed the quality filter.")
    print("  Suggestions:")
    print("  1. Set IMAGE_FILTER_CONFIG['landscape_only'] = False to allow portrait images")
    print("  2. Lower IMAGE_FILTER_CONFIG['center_weight_ratio'] (e.g., to 0.30)")
    print("  3. Lower IMAGE_FILTER_CONFIG['contrast_threshold'] (e.g., to 0.10)")
    print("  4. Add more external images that meet the criteria")
    exit()

# =============================================================================
# ONLINE RL LOOP
# =============================================================================
print("\n" + "="*80)
print("ONLINE RL: PROCESSING ALL IMAGES")
print("="*80)

approx_configs = [c for c in MAC_CONFIGURATIONS if c != 'mul8s_1KV6_add16se_2TN']
bandit         = ContextualBandit(approx_configs, tau=0.5)

image_database = []
output_dir     = '/content/online_rl_results'
os.makedirs(output_dir, exist_ok=True)

# Grad-CAM focus settings - DISABLED to show natural model attention
FOCUS_ON_REGION = False  # Let Grad-CAM show what the model naturally focuses on
FOCUS_STRENGTH  = 0.97   # Not used when FOCUS_ON_REGION = False
FOCUS_MODE      = 'hard'  # Not used when FOCUS_ON_REGION = False

# Accurate baseline config name
ACCURATE_CONFIG = 'mul8s_1KV6_add16se_2TN'

for img_idx, img_data in enumerate(all_images):

    img_array     = img_data['array']
    img_rgb_u8    = img_data['img_rgb_uint8']
    img_path      = img_data['path']
    img_source    = img_data['source']
    original_size = img_data['original_size']
    class_name    = img_data['class_name']
    quality       = img_data.get('quality_metrics', {})

    print(f"\n{'='*80}")
    print(f"IMAGE {img_idx + 1}/{len(all_images)}: {class_name} ({img_source})")
    print(f"  Size: {original_size}  |  "
          f"contrast={quality.get('rms_contrast','?')}  "
          f"centre={quality.get('center_weight_ratio','?')}")
    print('='*80)

    # Pre-process image tensor (ImageNet normalisation for MobileNetV2)
    _norm_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])
    img_tensor = _norm_tf(
        Image.fromarray(img_rgb_u8).resize((224, 224), Image.BILINEAR)
    ).to(device)

    prefix = f'img{img_idx}_{class_name.replace(" ", "_")}'

    # =========================================================================
    # ACCURATE GRAD-CAM — runs for every image (reference + first-image output)
    # =========================================================================
    print(f"\n🎯 Computing accurate Grad-CAM (mre=0.00)…")

    _em_acc      = MACEnergyModel(ACCURATE_CONFIG)
    _tracker_acc = LayerEnergyTracker(_em_acc)
    _wrapped_acc = MobileNetV2WithEnergyTracking(model, _tracker_acc)

    t0 = time.time()
    cam_acc_224, cam_acc_orig, tgt_cls, pred_cls, pred_conf, energy_acc = compute_grad_cam(
        _wrapped_acc, img_tensor, _tracker_acc,
        mre=MAC_CONFIGURATIONS[ACCURATE_CONFIG]['mre'],
        target_class=None,
        focus_on_region=FOCUS_ON_REGION,
        focus_strength=FOCUS_STRENGTH,
        focus_mode=FOCUS_MODE,
        img_rgb_uint8=img_rgb_u8,
    )
    elapsed_acc = time.time() - t0
    _wrapped_acc.cleanup_hooks()

    print(f"  ✓ Model prediction: class {pred_cls} with {pred_conf*100:.1f}% confidence")
    print(f"  ✓ Grad-CAM computed in {elapsed_acc:.3f}s, energy {energy_acc['total_energy_mJ']:.6f} mJ")
    print(f"  ✓ Heatmap stats: min={cam_acc_224.min():.3f}, max={cam_acc_224.max():.3f}, mean={cam_acc_224.mean():.3f}")
    print(f"  ✓ ROI focusing: {'ENABLED' if FOCUS_ON_REGION else 'DISABLED (showing natural model attention)'}")

    current_features = extract_gradient_features(cam_acc_224)

    # Save accurate Grad-CAM outputs (both unfocused and focused versions)
    img_pil_orig = Image.open(img_path).convert('RGB')
    img_pil_orig.save(os.path.join(output_dir, f'{prefix}_original.png'))

    # Save UNFOCUSED version (natural Grad-CAM)
    cv2.imwrite(
        os.path.join(output_dir, f'{prefix}_accurate_heatmap_NATURAL.png'),
        create_heatmap_jet(cam_acc_224, original_size),
    )
    cv2.imwrite(
        os.path.join(output_dir, f'{prefix}_accurate_overlay_NATURAL.png'),
        create_heatmap_overlay(cam_acc_224, img_rgb_u8, original_size),
    )

    # Save FOCUSED version if ROI focusing was applied
    if FOCUS_ON_REGION and cam_acc_orig is not cam_acc_224:
        cv2.imwrite(
            os.path.join(output_dir, f'{prefix}_accurate_heatmap_FOCUSED.png'),
            create_heatmap_jet(cam_acc_orig, original_size),
        )
        cv2.imwrite(
            os.path.join(output_dir, f'{prefix}_accurate_overlay_FOCUSED.png'),
            create_heatmap_overlay(cam_acc_orig, img_rgb_u8, original_size),
        )
        print("  ✓ Saved both NATURAL and FOCUSED Grad-CAM maps")
    else:
        print("  ✓ Saved NATURAL Grad-CAM maps (ROI focusing disabled)")

    # =========================================================================
    # FIRST IMAGE — accurate only; establishes reference for RL
    # =========================================================================
    if img_idx == 0:
        reference_cam      = cam_acc_224
        reference_features = current_features

        print(f"\n📊 Reference Heatmap Features:")
        print(f"  Mean Intensity : {reference_features['mean_intensity']:.4f}")
        print(f"  Sparsity       : {reference_features['sparsity']:.4f}")

        image_database.append({
            'image_idx':          img_idx,
            'image_path':         img_path,
            'source':             img_source,
            'class_name':         class_name,
            'original_size':      original_size,
            'reference_cam':      reference_cam,
            'reference_features': reference_features,
            'pred_class':         pred_cls,
            'pred_conf':          pred_conf,
            'quality_metrics':    quality,
            'mode':               'first_image_accurate_only',
        })
        print("\n✓ First image complete (reference established)")
        continue

    # =========================================================================
    # SUBSEQUENT IMAGES — RL-guided approximate Grad-CAM
    # =========================================================================
    is_similar = are_features_similar(current_features, reference_features)
    print(f"\n🔍 Similar to reference? {'YES' if is_similar else 'NO'}")

    if is_similar:
        context_features = reference_features
        selected_config  = bandit.select_action(context_features, mode='greedy')
        print("  Bandit mode: GREEDY")
    else:
        context_features = current_features
        selected_config  = bandit.select_action(context_features, mode='softmax')
        print("  Bandit mode: SOFTMAX")

    print(f"\n  Selected: {selected_config} "
          f"(Level {MAC_CONFIGURATIONS[selected_config]['level']}  "
          f"MRE {MAC_CONFIGURATIONS[selected_config]['mre']:.2f}%)")

    # Load the C++ approximate multiplier kernel for this config
    multiplier   = MAC_CONFIGURATIONS[selected_config]['multiplier']
    axx_kernel   = load_approximate_multiplier_kernel(multiplier)   # C++ kernel loaded here
    mre_selected = MAC_CONFIGURATIONS[selected_config]['mre']

    print(f"\n🔄 Computing approximate Grad-CAM with {multiplier}…")

    _em_approx      = MACEnergyModel(selected_config)
    _tracker_approx = LayerEnergyTracker(_em_approx)
    _wrapped_approx = MobileNetV2WithEnergyTracking(model, _tracker_approx)

    t0 = time.time()
    cam_approx_224, cam_approx_orig, _, _, _, energy_approx = compute_grad_cam(
        _wrapped_approx, img_tensor, _tracker_approx,
        mre=mre_selected,
        target_class=tgt_cls,
        focus_on_region=FOCUS_ON_REGION,
        focus_strength=FOCUS_STRENGTH,
        focus_mode=FOCUS_MODE,
        img_rgb_uint8=img_rgb_u8,
    )
    elapsed_approx = time.time() - t0
    _wrapped_approx.cleanup_hooks()

    print(f"  Grad-CAM time {elapsed_approx:.3f}s  "
          f"energy {energy_approx['total_energy_mJ']:.6f} mJ")

    # ── metrics ──────────────────────────────────────────────────────────────
    spearman_corr      = calculate_spearman_correlation(reference_cam, cam_approx_224)
    top_pixel_conc     = calculate_top_pixel_concentration(cam_approx_224)
    baseline_top_pixel = calculate_top_pixel_concentration(reference_cam)
    top_pixel_sim      = 1.0 - abs(top_pixel_conc - baseline_top_pixel)
    energy_efficiency  = 1.0 - (all_energy_mJ[selected_config] / E_max)

    reward = (
        REWARD_WEIGHTS['energy_efficiency']       * energy_efficiency +
        REWARD_WEIGHTS['spearman_correlation']    * max(spearman_corr, 0) +
        REWARD_WEIGHTS['top_pixel_concentration'] * max(top_pixel_sim,  0)
    )

    print(f"\n📊 Metrics:")
    print(f"  Spearman Corr  : {spearman_corr:.4f}")
    print(f"  Top-Pixel Conc : {top_pixel_conc:.4f}")
    print(f"  Energy Eff.    : {energy_efficiency:.4f}")
    print(f"  Reward         : {reward:.4f}")

    bandit.update(context_features, selected_config, reward)

    # ── save approximate Grad-CAM heatmap + overlay ───────────────────────
    print(f"\n🎨 Saving approximate Grad-CAM heatmaps…")

    # Save NATURAL version
    cv2.imwrite(
        os.path.join(output_dir, f'{prefix}_{selected_config}_heatmap_NATURAL.png'),
        create_heatmap_jet(cam_approx_224, original_size),
    )
    cv2.imwrite(
        os.path.join(output_dir, f'{prefix}_{selected_config}_overlay_NATURAL.png'),
        create_heatmap_overlay(cam_approx_224, img_rgb_u8, original_size),
    )

    # Save FOCUSED version if ROI focusing was applied
    if FOCUS_ON_REGION and cam_approx_orig is not cam_approx_224:
        cv2.imwrite(
            os.path.join(output_dir, f'{prefix}_{selected_config}_heatmap_FOCUSED.png'),
            create_heatmap_jet(cam_approx_orig, original_size),
        )
        cv2.imwrite(
            os.path.join(output_dir, f'{prefix}_{selected_config}_overlay_FOCUSED.png'),
            create_heatmap_overlay(cam_approx_orig, img_rgb_u8, original_size),
        )
        print("  ✓ Saved both NATURAL and FOCUSED approximate Grad-CAM maps")
    else:
        print("  ✓ Saved NATURAL approximate Grad-CAM maps")

    image_database.append({
        'image_idx':        img_idx,
        'image_path':       img_path,
        'source':           img_source,
        'class_name':       class_name,
        'original_size':    original_size,
        'selected_config':  selected_config,
        'pred_class':       pred_cls,
        'pred_conf':        pred_conf,
        'current_features': current_features,
        'quality_metrics':  quality,
        'metrics': {
            'spearman_correlation':    spearman_corr,
            'top_pixel_concentration': top_pixel_conc,
            'energy_efficiency':       energy_efficiency,
            'reward':                  reward,
        },
        'is_similar': is_similar,
        'mode':       'rl_selection',
    })

# =============================================================================
# FINAL SUMMARY
# =============================================================================
print("\n" + "="*80)
print("ONLINE RL COMPLETE — FINAL SUMMARY")
print("="*80)

bandit.print_summary()

COL = 160
print("\n" + "="*COL)
print("IMAGE PROCESSING RESULTS — OXFORD-IIIT PETS DATASET — MobileNetV2")
print("="*COL)
print(f"{'Img':<5} {'Class':<25} {'Source':<12} {'Config':<30} "
      f"{'Spearman':<10} {'Reward':<10} {'Contrast':<10} {'Centre':<10} {'Size':<15}")
print("-"*COL)

for d in image_database:
    size_str     = f"{d['original_size'][0]}x{d['original_size'][1]}"
    q            = d.get('quality_metrics', {})
    contrast_str = f"{q.get('rms_contrast', 0):.4f}"
    centre_str   = f"{q.get('center_weight_ratio', 0):.4f}"

    if d['mode'] == 'first_image_accurate_only':
        print(f"{d['image_idx']:<5} {d['class_name']:<25} {d['source']:<12} "
              f"{'Accurate (reference)':<30} {'1.0000':<10} {'--':<10} "
              f"{contrast_str:<10} {centre_str:<10} {size_str:<15}")
    else:
        m = d['metrics']
        print(f"{d['image_idx']:<5} {d['class_name']:<25} {d['source']:<12} "
              f"{d['selected_config']:<30} "
              f"{m['spearman_correlation']:<10.4f} {m['reward']:<10.4f} "
              f"{contrast_str:<10} {centre_str:<10} {size_str:<15}")

print("="*COL)

print(f"\n✅ COMPLETE!")
print(f"💾 Grad-CAM heatmaps saved to: {output_dir}")
print(f"\n📁 Per image outputs:")
print(f"  • *_original.png                      — original image")
print(f"  • *_accurate_heatmap_NATURAL.png      — natural Grad-CAM (what model sees)")
print(f"  • *_accurate_overlay_NATURAL.png      — natural Grad-CAM overlay")
if FOCUS_ON_REGION:
    print(f"  • *_accurate_heatmap_FOCUSED.png      — ROI-focused Grad-CAM")
    print(f"  • *_accurate_overlay_FOCUSED.png      — ROI-focused overlay")
print(f"  • *_<config>_heatmap_NATURAL.png      — RL-selected approx (natural)")
print(f"  • *_<config>_overlay_NATURAL.png      — RL-selected approx overlay")
if FOCUS_ON_REGION:
    print(f"  • *_<config>_heatmap_FOCUSED.png      — RL-selected approx (focused)")
    print(f"  • *_<config>_overlay_FOCUSED.png      — RL-selected approx overlay")
print(f"\n💡 ROI Focusing: {'ENABLED' if FOCUS_ON_REGION else 'DISABLED'}")
print(f"   Compare NATURAL vs FOCUSED versions to see the difference!")
print("\n🗂  Dataset: Oxford-IIIT Pet Dataset (37 cat & dog breeds)")
print("🤖 Model:   MobileNetV2 + AdaPT_Linear_Python classifier")
print("🔬 XAI:     Grad-CAM with approximate multiplier gradient perturbation")
print("="*80)


ONLINE RL — OXFORD-IIIT PETS DATASET — MOBILENETV2 — GRAD-CAM HEATMAPS

📋 Copying C++ files…
✓ C++ files copied

📦 Loading MobileNetV2 model from /content/mobilenetv2_checkpoint.pth…
  ℹ Detected num_classes = 10 from checkpoint ('model.classifier.1.weight')
  ⚠ Checkpoint has 10 output classes (not 102).
    Model will run as-is. To use a Oxford Pets model, retrain with num_classes=37.
✓ MobileNetV2 loaded  (num_classes=10)
✓ Extracted 53 layers

📊 Computing energy normalisation values…
  E_max (accurate baseline) = 2.757293 mJ

📸 Loading Oxford-IIIT Pets images…
📥 Loading Oxford-IIIT Pet Dataset…
  ✓ torchvision.datasets.OxfordIIITPet loaded  (3669 images, 37 breeds)
✓ Loaded 100 images from Oxford-IIIT Pets
  Breeds: Abyssinian, American Bulldog, American Pit Bull Terrier, Basset Hound, Beagle, Bengal, Bombay, Boxer, British Shorthair, Chihuahua, Egyptian Mau, English Cocker Spaniel, German Shorthaired, Great Pyrenees, Havanese, Japanese Chin, Keeshond, Leonberger, Maine Coon, Newf

ImportError: /root/.cache/torch_extensions/py312_cpu/PyInit_approx_mult_mul8s_1KVP/PyInit_approx_mult_mul8s_1KVP.so: cannot open shared object file: No such file or directory